In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
# input = r"/Users/cmdb/Quant_Bio_Project/Quant-Bio-Project/segmentation_test.tif" 
# img = cv.imread(input, cv.IMREAD_GRAYSCALE)



In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

def gaussian_kernal(size,std):
    kernel = np.fromfunction(
        lambda x,y: np.divide(1,2*np.pi* std**2) * 
        np.exp(
            -((x-(size-1)/2)**2 + (y-(size-1)/2)**2) / (2* std**2)
               ),
        (size,size)
    )
    return np.array(kernel/np.sum(kernel))

def find(frame,mask,contour,pixelSize):

    """
    This function takes in an image and draws the given contours.  It returns an inverted boolean array
    where true = pixel within a contour. This gives the number of pixels within a given contour to calculate area, as well as the 
    starting point for blob labeling and tracking.
    """
    #dont want to edit the original image
    filled_mask = np.copy(mask)
    #draw the contour
    filled_mask = cv.drawContours(filled_mask,[contour], 0,(0, 255, 0),thickness=cv.FILLED)
    #boolean mask and returning it, ensuring true = 0, false = 1. The sum of this is pixel size. 
    image_mask = filled_mask > 0
    total_pixels = np.sum(~image_mask.astype(int))
    image_mask = ~image_mask
    total_pixels = np.sum(image_mask.astype(int))
    #output [frame, x,y, Area, TotalIntensity]
    total_intensity = np.sum(mask*(image_mask.astype(int)))
    ##Find X,Y center of mass based on contours. 
    #https://docs.opencv.org/3.4/dd/d49/tutorial_py_contour_features.html
    M = cv.moments(contour)
    x = int(M['m10']/M['m00'])
    y = int(M['m01']/M['m00'])
    #in units of pixelSize(micron squared)
    total_area = total_pixels*(pixelSize**2)
    return [frame,[x,y],total_area,total_intensity]
kernel = gaussian_kernal(3,np.sqrt(3))
low_thresh: int = 13
high_thresh: int = 50

#contour filtering
# we want anything equal to or greater than 5
filter_val = 10

##Pixel Size. This is used for area calculations. Where the area of a pixel is pixelSize**2
pixel_size: float = 0.115 


## Apply pixel normalization

Min Pixel Value = min(image)
Max Pixel Value = max(image)

Normalized = {pixel - min pixel}/{max pixel - min pixel}

In [ ]:
# contrs, hierch = cv.findContours(thresh,1,2)

Next? Check contrs output

In [ ]:
# epsilon = cv.arcLength(cnt,True)
# approx = cv.approxPolyDP(cnt,epsilon,True)

## Time to work on the full stack analysis

#### Working with a stacked tiff

Implemented

## Convert from uint16 to uint8
CV2 threshold function only works with uint8 values, so we have to convert it. This is done by dividing it by 256 and assigning it as a uint8 datatype. 

## Histogram Normalization

In [ ]:
# image1 = cv.filter2D(tiff_stack[10],-1,kernel)
# plt.imshow(image1)
# plt.show()

Converted stacked tiff into opencv 

In [ ]:
import skimage.io as io
tiff_stack = io.imread("/Users/cmdb/Quant_Bio_Project/Quant-Bio-Project/cell2.tif",plugin='tifffile')
tiff_stack = (tiff_stack/256).astype(np.uint8)
copied = np.copy(tiff_stack)
frame_dict = {}
for i in range(len(tiff_stack)):
    ret,thresh = cv.threshold(tiff_stack[i,:,:],low_thresh,high_thresh,cv.THRESH_BINARY)
    contours,hierarchy = cv.findContours(thresh, 1, 2)
    #filtering out contours less than 10
    contours = [lst for lst in contours if len(lst) >= filter_val]
    frame_dict[f"Timepoint {i}"] = []
    for j in contours:
        #thresholding step needed to skip bad areas. Area == 0 is nothing. 
        area = cv.moments(j)["m00"]
        if area > 0:
            centroid_output = find(i,copied[i,:,:],j,pixel_size)
            frame_dict[f"Timepoint {i}"].append(centroid_output)


## Tracking by Elucidian Distance

In [35]:
##Challenges
# iterate through the dictionary by frame ##### success
# iterate through the dictionary by frame and frame + 1 ##### success
# pull x,y coords ##### success
# create blobs for n, pair n+1 with n blobs (first two frames only)

def elucidian_distance(n_coords,n1_coords):
   # based on 2d distance formula of sqrt((x2-x1)**2 + (y2-y1)**2))
   x1 = n_coords[0]
   y1 = n_coords[1]
   x2 = n1_coords[0]
   y2 = n1_coords[1]
   distance = np.sqrt((x2-x1)**2 + (y2-y1)**2)
   return distance

def search_matching_by_distance(blobs,n_coords,n1_coords):
   


   pass

def blob_tracking_simple(frame,dict,n,n1):
   #n denotes 
   for i in range(len(n)):
      #this assigns the first frame centroids as blobs i+1
      if frame == 0:
         dict[f"Blob {i+1}"] = [n[i]]
   print(dict)
   pass
   

#keys list
list_of_keys = sorted(frame_dict.keys())
#this is where we will be storing the tracked blobs
blobs: dict = {}
for i in range(1):
   n = list_of_keys[i]
   n1 = list_of_keys[i+1]
   coords_n = frame_dict[n]
   coords_n1 = frame_dict[n1]
   
   blob_tracking_simple(i,blobs,coords_n,coords_n1)
   pass

{'Blob 1': [[0, [127, 166], np.float64(18.938200000000002), np.int64(168)]], 'Blob 2': [[0, [134, 154], np.float64(18.9911), np.int64(251)]], 'Blob 3': [[0, [133, 149], np.float64(19.01755), np.int64(274)]], 'Blob 4': [[0, [147, 140], np.float64(19.0969), np.int64(385)]], 'Blob 5': [[0, [156, 126], np.float64(19.17625), np.int64(461)]], 'Blob 6': [[0, [152, 108], np.float64(20.168125), np.int64(2224)]], 'Blob 7': [[0, [149, 97], np.float64(19.17625), np.int64(568)]], 'Blob 8': [[0, [133, 83], np.float64(18.951425), np.int64(192)]]}
